In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import torch.optim as optim

In [8]:
X_raw = np.load('/kaggle/input/datasets/muhammadhashimtariq/datascience-groupproj/X_data.npy')
y_raw = np.load('/kaggle/input/datasets/muhammadhashimtariq/datascience-groupproj/y_data.npy')
y_raw = np.log1p(y_raw)
# 2. Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

Training samples: 33088, Testing samples: 8272


In [9]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

In [10]:
# Hyperparameters
BATCH_SIZE = 32

# Create Datasets
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

# Create DataLoaders
# We shuffle the training data to ensure the model doesn't learn the order of rows
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# No need to shuffle test data
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Quick Check
for batch_X, batch_y in train_loader:
    print(f"Batch X shape: {batch_X.shape}") # Should be [32, num_features]
    print(f"Batch y shape: {batch_y.shape}") # Should be [32, 1]
    break

Batch X shape: torch.Size([32, 36])
Batch y shape: torch.Size([32, 1])


In [17]:
class LongevityPredictor(nn.Module):
    def __init__(self, input_size):
        super(LongevityPredictor, self).__init__()

        self.fwd = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256) ,
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1)
            
        )
        
        
        # Dropout to prevent overfitting (ignores 20% of neurons randomly)
        self.dropout = nn.Dropout(0.2)

        self.relu = nn.ReLU()

    def forward(self, x):
        # Pass through Layer 1
        return self.fwd(x)

# Initialize the model
# X_train_t.shape[1] is the number of features we prepared earlier
input_dim = X_train_t.shape[1]
model = LongevityPredictor(input_dim)

print(model)

LongevityPredictor(
  (fwd): Sequential(
    (0): Linear(in_features=36, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): Linear(in_features=64, out_features=32, bias=True)
    (10): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU()
    (12): Linear(in_features=32, out_features=1, bias=True)
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (relu): ReLU()
)


In [18]:
LR = 0.001
EPOCHS = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [19]:

epochs = 50

model.to(device)

train_losses = []
val_losses = []

# 2. The Loop
for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    batch_losses = []
    
    # tqdm wrapper for the progress bar
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
    
    for batch_X, batch_y in train_bar:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward pass
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        batch_losses.append(loss.item())
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = sum(batch_losses) / len(batch_losses)
    train_losses.append(avg_train_loss)

    # --- VALIDATION PHASE ---
    model.eval()
    val_batch_losses = []
    
    with torch.no_grad():
        for val_X, val_y in test_loader:
            val_X, val_y = val_X.to(device), val_y.to(device)
            val_preds = model(val_X)
            val_loss = criterion(val_preds, val_y)
            val_batch_losses.append(val_loss.item())

    avg_val_loss = sum(val_batch_losses) / len(val_batch_losses)
    val_losses.append(avg_val_loss)

    # Print summary for the epoch
    print(f"Epoch {epoch+1:02d}: Train Loss = {avg_train_loss:.4f} | Val Loss = {avg_val_loss:.4f}")

Epoch 1/50 [Train]:   0%|          | 0/1034 [00:00<?, ?it/s]

Epoch 01: Train Loss = 22.8805 | Val Loss = 2.2836


Epoch 2/50 [Train]:   0%|          | 0/1034 [00:00<?, ?it/s]

Epoch 02: Train Loss = 2.2763 | Val Loss = 2.2638


Epoch 3/50 [Train]:   0%|          | 0/1034 [00:00<?, ?it/s]

...

Epoch 48: Train Loss = 1.8604 | Val Loss = 1.8789


Epoch 49/50 [Train]:   0%|          | 0/1034 [00:00<?, ?it/s]

Epoch 49: Train Loss = 1.8633 | Val Loss = 1.8976


Epoch 50/50 [Train]:   0%|          | 0/1034 [00:00<?, ?it/s]

Epoch 50: Train Loss = 1.8604 | Val Loss = 1.8775
